4- Results, Lift & Business impact

Present findings clearly and quantify the business value of rolling out of the new ad.

4.1 Setup

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint

df = pd.read_csv('../data/marketing_AB_clean.csv')

groups = df.groupby('test group').agg(
    n=('converted', 'count'),
    conversions=('converted', 'sum'),
    rate=('converted', 'mean')
)
p_ad  = groups.loc['ad',  'rate'] if 'ad'  in groups.index else groups['rate'].iloc[0]
p_psa = groups.loc['psa', 'rate'] if 'psa' in groups.index else groups['rate'].iloc[1]
n_ad  = groups.loc['ad',  'n']    if 'ad'  in groups.index else groups['n'].iloc[0]
n_psa = groups.loc['psa', 'n']    if 'psa' in groups.index else groups['n'].iloc[1]

4.2 Confidence interval for the difference

In [2]:
from statsmodels.stats.proportion import confint_proportions_2indep

low, high = confint_proportions_2indep(
    groups['conversions'].iloc[0], groups['n'].iloc[0],
    groups['conversions'].iloc[1], groups['n'].iloc[1],
    method='wald'
)
diff = p_ad - p_psa
print(f"Conversion rate - ad  : {p_ad*100:.3f}%")
print(f"Conversion rate -psa  : {p_psa*100:.3f}%")
print(f"Absolute difference   : {diff*100:+.3f} pp")
print(f"95% CI for difference : [{low*100:.3f}pp, {high*100:.3f}pp]")

Conversion rate - ad  : 2.555%
Conversion rate -psa  : 1.785%
Absolute difference   : +0.769 pp
95% CI for difference : [0.595pp, 0.943pp]


4-3 Lift & projction to 100% of users

In [3]:
total_users = len(df)
current_conversions = int(groups['conversions'].sum())

# If everyone saw the new ad 
projected_conversions = int(p_ad * total_users)
lift_abs = projected_conversions - current_conversions
lift_pct = (p_ad - p_psa) / p_psa * 100

print(f"Total users in dataset   : {total_users:,}")
print(f"Current conversions)    : {current_conversions:,}")
print(f"Projected (100% new ad) : {projected_conversions:,}")
print(f"Additional conversions) : {lift_abs:,}")
print(f"Relative lift           : {lift_pct:,}")


Total users in dataset   : 588,101
Current conversions)    : 14,843
Projected (100% new ad) : 15,023
Additional conversions) : 180
Relative lift           : 43.085064022225836


4-4 Visualize the lift with confidence intervals

In [4]:
# CIs for each group rate
ci_ad = proportion_confint(groups['conversions'].iloc[0], groups['n'].iloc[0], method='wilson')
ci_psa = proportion_confint(groups['conversions'].iloc[1], groups['n'].iloc[1], method='wilson')

lables = ['Control (psa)', 'Test (ad)']
means = [p_psa *100, p_ad * 100]
errors = [
    [(p_psa - ci_psa[0])*100, (ci_psa[1] - p_psa)*100],
    [(p_ad - ci_ad[0])*100, (ci_ad[1] - p_ad)*100],
]
yerr = np.array(errors).T

fig,ax = plt.subplots(figsize=(7, 5))
colors = ['#5B6EE1' ,'#E15B5B']
bars = ax.bar(lables, means, color=colors, width=0.4,
              yerr=yerr, capsize=10, error_kw={'linewidth':2, 'ecolor':'#333'})

for bar, val in zip(bars,means):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.15,
            f"{val:.2f}%" , ha='center', fontsize=12, fontweight='bold')

ax.annotate(f"LIFT: {lift_pct:+.1f}%",
            xy=(1, p_ad*100), xytext=(1.35, (p_ad+p_psa)/2*100),
            arrowprops=dict(arrowstyle='->' , color='#333'),
            fontsize=11, color='#333')

ax.set_ylabel('Conversion rate (%)' , fontsize=12)
ax.set_title('Lift: new ad vs public announcement' , fontsize=13)
ax.set_ylim(0, max(means)*1.6)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y' , linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('../outputs/plots/lift_chart.png' , dpi=150)
plt.show()

<Figure size 700x500 with 1 Axes>

###### Written conclusion

The A/B test compared conversion rates between the ad group (N = 564577) 
and the psa control group (N =23524). 

The ad group achieved a conversion rate of 2.555% vs 1.785% for the control 
(absolute difference: 0.769pp, 95% CI [0.595, 0.943]pp).

A two-proportion Z-test yielded z =7.3701, p = 0.000000 < 0.05.

Conclusion: Reject  H₀. The new ad did
produce a statistically significant improvement in conversion rate.

Business impact: if rolled out to 100% of users, the new ad would 
generate approximately 4.524 additional conversions (+43.1% lift).

